## Citi Bike Prediction Project — Data Acquisition and Preprocessing

This notebook assembles the base dataset for the Citi Bike low dock availability prediction project.

The workflow combines static Citi Bike station metadata, time-based station status snapshots, and MTA subway station data to create a cleaned station-level dataset for downstream exploratory analysis and modeling. A key objective of this notebook is to construct the short-term prediction target by estimating whether a station will experience low dock availability approximately 30 minutes in the future.

#### Load libraries

This section imports the core Python libraries used throughout the notebook for data loading, API requests, file handling, and preprocessing.

The workflow in this notebook combines:
- Citi Bike station metadata from the GBFS API
- Citi Bike station status snapshots collected over time
- MTA subway station data for transit-access feature engineering

These imports support the full data acquisition and early preprocessing pipeline.

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np
import datetime
import time

# File path locations
from pathlib import Path

# Load data pull libraries
import requests
import json

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Improve plot styling
sns.set()

# Show all columns
pd.set_option('display.max_columns', None)

#### Load data directories

This section defines the project file paths used to read raw inputs and save cleaned outputs.

Keeping directory paths centralized helps make the notebook more reproducible and easier to maintain as the project grows across preprocessing, exploratory analysis, feature engineering, and modeling.

In [2]:
# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import and inspect `station_information`

This section loads Citi Bike station metadata and performs an initial quality check.

The station information dataset contains relatively static attributes such as station name, latitude, longitude, capacity, and region. These fields will later be used to:
- identify valid NYC stations
- support geospatial feature engineering
- provide station-level context for the prediction dataset

Initial inspection helps confirm schema, missingness, and station coverage before any filtering is applied.

In [3]:
# Load 'station_information' data from API
station_info_url = "https://gbfs.lyft.com/gbfs/1.1/bkn/en/station_information.json"
response = requests.get(station_info_url, timeout=10)
response.raise_for_status()

payload = response.json()
rows = payload["data"]["stations"]

# Import into a dataframe
station_info = pd.DataFrame(rows)

# Save as parquet file
station_info.to_parquet(RAW_DATA_DIR / "01_station_info.parquet")

In [4]:
# Load data saved as a parquet file
station_info = pd.read_parquet(
    RAW_DATA_DIR / "01_station_info.parquet",
    columns = ["station_id","name","lat","lon","capacity", "region_id"]
)

In [5]:
# Show first five records of data
station_info.head(5)

,station_id,name,lat,lon,capacity,region_id
0,1905837242740508940,31 St & Broadway,40.762110,-73.925230,35,71
1,41495491-5d89-4e14-aab9-c3db04aad399,43 St & Skillman Ave,40.746927,-73.920825,19,71
2,2177014969129222184,Henry Hudson Pkwy E & W 231 St,40.883360,-73.915050,0,71
3,2177016530523622712,Post Rd & W 251 St,40.895610,-73.897930,0,71
4,2177016940364559212,David Sheridan Plaza & Broadway,40.905010,-73.896630,0,71


In [6]:
# Number of rows and columns
station_info.shape

(2360, 6)

In [7]:
# Data types along with count of missing
station_info_types = pd.DataFrame()
station_info_types["Data Type"] = station_info.dtypes
station_info_types["Count of Nulls"] = station_info.isna().sum()
station_info_types

,Data Type,Count of Nulls
station_id,object,0
name,object,0
lat,float64,0
lon,float64,0
capacity,int64,0
region_id,object,11


In [8]:
# unique station_ids
station_info["station_id"].nunique()

2360

#### Filter and standardize `station_information`

This section narrows the station metadata to valid New York City Citi Bike stations and removes records that would not be useful for modeling.

Key cleanup steps include:
- restricting the data to NYC stations only
- removing stations with invalid or zero capacity
- renaming location fields for clarity and consistency

These steps ensure that only usable station records are carried forward into the modeling dataset.

In [9]:
# Keep only 'region_id' = 71 as this is NYC
station_info = station_info[station_info["region_id"] == "71"].copy()
station_info.shape

(2246, 6)

In [10]:
# Keep only stations where capacity is > 0
station_info = station_info[station_info["capacity"] > 0].copy()
station_info.shape

(2221, 6)

In [11]:
# Rename "lat" and "lon" columns
station_info = station_info.rename(columns={
    "name":"citi_bike_station_name",
    "lat":"citi_bike_lat",
    "lon":"citi_bike_lon"
}).copy()

station_info.columns

Index(['station_id', 'citi_bike_station_name', 'citi_bike_lat',
       'citi_bike_lon', 'capacity', 'region_id'],
      dtype='object')

#### MTA station data

This section loads and cleans MTA subway station data that will later be used to create a transit-access feature for each Citi Bike station.

Because subway proximity may influence Citi Bike demand and dock availability, this dataset is incorporated as a geospatial enrichment source. The preparation here focuses on:
- standardizing column names
- reducing the data to the fields needed for spatial matching
- preparing the data for nearest-station calculations

In [12]:
mta = pd.read_parquet(RAW_DATA_DIR / "00_mta_subway_stations.parquet")

In [13]:
mta.head()

,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference
0,127,317,611,IRT,Broadway - 7Av,Times Sq-42 St,M,True,1 2 3,Subway,40.755290,-73.987495,Uptown,Downtown,1,1,1,None,POINT (-73.987495 40.75529)
1,S17,515,515,SIR,Staten Island,Annadale,SI,False,SIR,Open Cut,40.540460,-74.178217,Ferry,South Shore,0,0,0,None,POINT (-74.178217 40.54046)
2,S01,139,627,BMT,Franklin Shuttle,Franklin Av,Bk,False,S,Elevated,40.680596,-73.955827,Last Stop,Prospect Park,1,1,1,None,POINT (-73.955827 40.680596)
3,254,349,349,IRT,Eastern Pky,Junius St,Bk,False,3,Elevated,40.663515,-73.902447,Manhattan,New Lots,0,0,0,None,POINT (-73.902447 40.663515)
4,M01,108,108,BMT,Myrtle Av,Middle Village-Metropolitan Av,Q,False,M,Elevated,40.711396,-73.889601,Inbound,Last Stop,1,1,1,None,POINT (-73.889601 40.711396)


In [14]:
# Make column names consistent format
mta.columns = (
    mta.columns.str.strip()
             .str.lower()
             .str.replace(" ", "_")
             .str.replace("-", "_")
)

mta.columns

Index(['gtfs_stop_id', 'station_id', 'complex_id', 'division', 'line',
       'stop_name', 'borough', 'cbd', 'daytime_routes', 'structure',
       'gtfs_latitude', 'gtfs_longitude', 'north_direction_label',
       'south_direction_label', 'ada', 'ada_northbound', 'ada_southbound',
       'ada_notes', 'georeference'],
      dtype='object')

In [15]:
# Reduce to only needed columns
mta_reduced = mta[["stop_name", 
                   "gtfs_latitude", 
                   "gtfs_longitude"
                  ]].copy()

mta_reduced.columns

Index(['stop_name', 'gtfs_latitude', 'gtfs_longitude'], dtype='object')

#### Import and inspect `station_status`

This section loads and compiles repeated Citi Bike station status snapshots collected over time.

Unlike station metadata, station status is dynamic and captures operational conditions such as:
- available bikes
- available docks
- reporting timestamps

These repeated snapshots form the time-based backbone of the project and are used to build the prediction target for low dock availability in the next 30 minutes.

In [16]:
# Import and compile all JSON files
json_files = sorted((RAW_DATA_DIR / "snapshots").rglob("*.json"))

snapshots_dfs = []

for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        snapshot = json.load(f)

    df = pd.DataFrame(snapshot["station_status"]["data"]["stations"])
    df["snapshot_time_utc"] = snapshot["snapshot_time_utc"]
    snapshots_dfs.append(df)

station_status_full_df = pd.concat(snapshots_dfs, ignore_index=True)

In [17]:
# Reduce to just five columns
station_status = station_status_full_df[[
    "station_id", 
    "num_bikes_available", 
    "num_docks_available", 
    "snapshot_time_utc", 
    "last_reported"
]].copy()

In [18]:
# Number of rows and columns
station_status.shape

(775920, 5)

In [19]:
# View first five rows
station_status.head(5)

,station_id,num_bikes_available,num_docks_available,snapshot_time_utc,last_reported
0,41495491-5d89-4e14-aab9-c3db04aad399,0,0,2026-03-22T03:56:42.814260+00:00,86400
1,1905837242740508940,0,0,2026-03-22T03:56:42.814260+00:00,86400
2,2177014969129222184,0,0,2026-03-22T03:56:42.814260+00:00,86400
3,2177016530523622712,0,0,2026-03-22T03:56:42.814260+00:00,86400
4,2177016940364559212,0,0,2026-03-22T03:56:42.814260+00:00,86400


In [20]:
# Data types along with count of missing
station_status_dtypes = pd.DataFrame()
station_status_dtypes["Data Type"] = station_status.dtypes
station_status_dtypes["Count of Nulls"] = station_status.isna().sum()
station_status_dtypes

,Data Type,Count of Nulls
station_id,object,0
num_bikes_available,int64,0
num_docks_available,int64,0
snapshot_time_utc,object,0
last_reported,int64,0


#### Clean timestamps and derive time features

This section standardizes the time-related fields in the station status data and creates early temporal features.

Because this project is built around short-term forecasting, timestamp quality is especially important. The main goals here are to:
- convert raw reporting fields into usable datetime values
- remove rows with invalid or missing timestamps
- derive basic time features such as hour of day and weekday

These variables will later support both exploratory analysis and predictive modeling.

In [21]:
# Make 'last_reported' numeric field
station_status["last_reported"] = pd.to_numeric(station_status["last_reported"], errors="coerce")

# Make datetime anything above the year '2001'
threshold = 1000000000
station_status["timestamp"] = pd.to_datetime(
    station_status["last_reported"].where(station_status["last_reported"] > threshold), 
    errors="coerce",
    unit="s"
)

station_status.head()

,station_id,num_bikes_available,num_docks_available,snapshot_time_utc,last_reported,timestamp
0,41495491-5d89-4e14-aab9-c3db04aad399,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT
1,1905837242740508940,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT
2,2177014969129222184,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT
3,2177016530523622712,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT
4,2177016940364559212,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT


In [22]:
# Make 'snapshot_time_utc' standard datetime
station_status["snapshot_time"] = (
    pd.to_datetime(station_status["snapshot_time_utc"], utc=True).dt.floor("s").dt.tz_localize(None)
)

station_status.head()

,station_id,num_bikes_available,num_docks_available,snapshot_time_utc,last_reported,timestamp,snapshot_time
0,41495491-5d89-4e14-aab9-c3db04aad399,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT,2026-03-22 03:56:42
1,1905837242740508940,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT,2026-03-22 03:56:42
2,2177014969129222184,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT,2026-03-22 03:56:42
3,2177016530523622712,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT,2026-03-22 03:56:42
4,2177016940364559212,0,0,2026-03-22T03:56:42.814260+00:00,86400,NaT,2026-03-22 03:56:42


In [23]:
# Check number of values where "timestamp" is 'NaT'
station_status[station_status["timestamp"].isna()].shape

(14411, 7)

In [24]:
# Drop rows where timestamp is unavailable
station_status = station_status.dropna(subset=["timestamp"]).copy()
station_status.shape

(761509, 7)

In [25]:
# Check number of values where "snapshot_time" is 'NaT'
station_status[station_status["snapshot_time"].isna()].shape

(0, 7)

In [26]:
# Drop rows where "snapshot_time" is unavailable
station_status = station_status.dropna(subset=["snapshot_time"]).copy()
station_status.shape

(761509, 7)

In [27]:
# Create hour, weekday columns
station_status["snapshot_hr"] = station_status["snapshot_time"].dt.hour
station_status["snapshot_weekday"] = station_status["snapshot_time"].dt.dayofweek

station_status[~station_status["timestamp"].isna()].head(5)

,station_id,num_bikes_available,num_docks_available,snapshot_time_utc,last_reported,timestamp,snapshot_time,snapshot_hr,snapshot_weekday
7,5645b05e-85be-460d-8506-cacd20bff233,0,0,2026-03-22T03:56:42.814260+00:00,1773760847,2026-03-17 15:20:47,2026-03-22 03:56:42,3,6
8,5526b855-dc09-4159-a20a-99651861a481,0,0,2026-03-22T03:56:42.814260+00:00,1773765268,2026-03-17 16:34:28,2026-03-22 03:56:42,3,6
10,2171903488884523500,9,1,2026-03-22T03:56:42.814260+00:00,1774151328,2026-03-22 03:48:48,2026-03-22 03:56:42,3,6
11,66dd056e-0aca-11e7-82f6-3863bb44ef7c,3,0,2026-03-22T03:56:42.814260+00:00,1774151366,2026-03-22 03:49:26,2026-03-22 03:56:42,3,6
12,2124036994734951288,11,1,2026-03-22T03:56:42.814260+00:00,1774151560,2026-03-22 03:52:40,2026-03-22 03:56:42,3,6


In [28]:
# Add a column for weekend indicator
weekend_function = lambda x : 1 if (x == 5 or  x == 6) else 0

station_status["is_weekend"] = station_status["snapshot_weekday"].apply(weekend_function)

station_status.head()

,station_id,num_bikes_available,num_docks_available,snapshot_time_utc,last_reported,timestamp,snapshot_time,snapshot_hr,snapshot_weekday,is_weekend
7,5645b05e-85be-460d-8506-cacd20bff233,0,0,2026-03-22T03:56:42.814260+00:00,1773760847,2026-03-17 15:20:47,2026-03-22 03:56:42,3,6,1
8,5526b855-dc09-4159-a20a-99651861a481,0,0,2026-03-22T03:56:42.814260+00:00,1773765268,2026-03-17 16:34:28,2026-03-22 03:56:42,3,6,1
10,2171903488884523500,9,1,2026-03-22T03:56:42.814260+00:00,1774151328,2026-03-22 03:48:48,2026-03-22 03:56:42,3,6,1
11,66dd056e-0aca-11e7-82f6-3863bb44ef7c,3,0,2026-03-22T03:56:42.814260+00:00,1774151366,2026-03-22 03:49:26,2026-03-22 03:56:42,3,6,1
12,2124036994734951288,11,1,2026-03-22T03:56:42.814260+00:00,1774151560,2026-03-22 03:52:40,2026-03-22 03:56:42,3,6,1


In [29]:
# Drop "snapshot_time_utc" and "last_reported"
station_status = station_status.drop(columns=["snapshot_time_utc", "last_reported"], axis=1)
station_status.columns

Index(['station_id', 'num_bikes_available', 'num_docks_available', 'timestamp',
       'snapshot_time', 'snapshot_hr', 'snapshot_weekday', 'is_weekend'],
      dtype='object')

#### Find nearest MTA station to each Citi Bike station

This section uses geospatial nearest-neighbor logic to calculate the closest subway station for each Citi Bike station.

Transit access may influence biking demand, especially around commuting corridors, so this step adds an interpretable location-based feature:
- nearest subway station
- distance to nearest subway station in meters

This feature is intended to enrich the forecasting dataset with nearby transit context.

In [30]:
# Import library
from sklearn.neighbors import BallTree

In [31]:
# Create Citi Bike radians
citi_bike_radians = np.radians(station_info[["citi_bike_lat", "citi_bike_lon"]])

In [32]:
# Create MTA subway station radians
mta_radians = np.radians(mta_reduced[["gtfs_latitude","gtfs_longitude"]])

In [33]:
# Build BallTree
mta_tree = BallTree(mta_radians, metric="haversine")

In [34]:
# Query for nearest station
mta_distances, mta_indices = mta_tree.query(citi_bike_radians, k=1)

In [35]:
# Earth radius in meters
earth_radius = 6371000

# Convert distances from radians to meters
mta_distance_m = mta_distances * earth_radius

In [36]:
# Add results back to dataframe
station_info["nearest_mta_station"] = mta_reduced["stop_name"].iloc[mta_indices.flatten()].values
station_info["meters_to_nearest_mta_station"] = mta_distance_m.flatten()

station_info.head()

,station_id,citi_bike_station_name,citi_bike_lat,citi_bike_lon,capacity,region_id,nearest_mta_station,meters_to_nearest_mta_station
0,1905837242740508940,31 St & Broadway,40.762110,-73.925230,35,71,Broadway,39.850265
1,41495491-5d89-4e14-aab9-c3db04aad399,43 St & Skillman Ave,40.746927,-73.920825,19,71,40 St-Lowery St,441.165293
5,06439006-11b6-44f0-8545-c9d39035f32a,Vesey St & Church St,40.712220,-74.010472,78,71,World Trade Center,70.844613
6,16e70b05-5b73-4930-9dcf-f79e5ce9eaf5,44 St & 3 Ave,40.651173,-74.011405,23,71,45 St,275.022135
8,5526b855-dc09-4159-a20a-99651861a481,17 St & 5 Ave,40.663493,-73.991007,25,71,Prospect Av,265.277299


#### Merge `station_information` and `station_status`

This section combines the static station metadata with the time-varying station status snapshots to create a unified station-level time series dataset.

Merging these sources allows each station-time observation to include:
- operational dock and bike availability
- station capacity and coordinates
- transit-access features

Post-merge checks are included to identify unmatched stations and remove records that are incomplete or not suitable for modeling.

In [37]:
station_data_merged = station_status.merge(
    station_info,
    on="station_id",
    how="left",
    suffixes=(None, None)
).sort_values(by=["station_id", "snapshot_time"], ascending=[True, True])

station_data_merged.head(5)

,station_id,num_bikes_available,num_docks_available,timestamp,snapshot_time,snapshot_hr,snapshot_weekday,is_weekend,citi_bike_station_name,citi_bike_lat,citi_bike_lon,capacity,region_id,nearest_mta_station,meters_to_nearest_mta_station
2094,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 03:53:40,2026-03-22 03:56:42,3,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959
4217,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 04:05:48,2026-03-22 04:07:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959
6589,00284700-9d22-42ce-8485-113fed9879c1,12,6,2026-03-22 04:34:07,2026-03-22 04:37:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959
8777,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:04:29,2026-03-22 05:07:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959
11360,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:36:51,2026-03-22 05:37:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959


In [38]:
# Check number of missing values on merged dataset
station_data_merged.isna().sum()

station_id                           0
num_bikes_available                  0
num_docks_available                  0
timestamp                            0
snapshot_time                        0
snapshot_hr                          0
snapshot_weekday                     0
is_weekend                           0
citi_bike_station_name           38460
citi_bike_lat                    38460
citi_bike_lon                    38460
capacity                         38460
region_id                        38460
nearest_mta_station              38460
meters_to_nearest_mta_station    38460
dtype: int64

In [39]:
# Check stations where capacity is NA
station_data_merged[station_data_merged["capacity"].isna()]["station_id"].nunique()

119

In [40]:
# Drop stations where capacity is NA
station_data_merged = station_data_merged.dropna(subset=["capacity"]).copy()
station_data_merged = station_data_merged[station_data_merged["capacity"] > 0].copy()

In [41]:
# Check number of rows, columns after drop
station_data_merged.shape

(723049, 15)

#### Dock availability

This section creates the current and future dock-availability measures that support the project target definition.

The workflow first calculates each station’s current dock availability as a share of station capacity, then uses within-station time ordering and row shifting to identify the next observed snapshot. This makes it possible to estimate dock availability approximately 30 minutes into the future.

Additional validation checks are included to:
- remove impossible values
- verify there are no invalid dock counts relative to station capacity
- confirm that shifted observations are actually close to 30 minutes ahead

In [42]:
# Compute available percentage per timestamp
station_data_merged["current_dock_pct"] = (
    station_data_merged["num_docks_available"] / station_data_merged["capacity"]
)

station_data_merged.head(5)

,station_id,num_bikes_available,num_docks_available,timestamp,snapshot_time,snapshot_hr,snapshot_weekday,is_weekend,citi_bike_station_name,citi_bike_lat,citi_bike_lon,capacity,region_id,nearest_mta_station,meters_to_nearest_mta_station,current_dock_pct
2094,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 03:53:40,2026-03-22 03:56:42,3,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053
4217,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 04:05:48,2026-03-22 04:07:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053
6589,00284700-9d22-42ce-8485-113fed9879c1,12,6,2026-03-22 04:34:07,2026-03-22 04:37:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.315789
8777,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:04:29,2026-03-22 05:07:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053
11360,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:36:51,2026-03-22 05:37:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053


In [43]:
# Drop rows where current dock count exceeds station capacity
station_data_merged = station_data_merged[
    station_data_merged["num_docks_available"] <= station_data_merged["capacity"]
].copy()

station_data_merged.shape

(721962, 16)

In [44]:
# Verify operation
(station_data_merged["num_docks_available"] > station_data_merged["capacity"]).sum()

0

In [45]:
# Check for duplicates before shifting
duplicate_station_time = station_data_merged.duplicated(
    subset=["station_id", "snapshot_time"]
).sum()

print(f"Duplicate station_id + snapshot_time rows: {duplicate_station_time}")

Duplicate station_id + snapshot_time rows: 0


In [46]:
# Shift future docks available within each station
station_data_merged["future_num_docks_available"] = (
    station_data_merged.groupby("station_id")["num_docks_available"].shift(-1)
)

station_data_merged.head(5)

,station_id,num_bikes_available,num_docks_available,timestamp,snapshot_time,snapshot_hr,snapshot_weekday,is_weekend,citi_bike_station_name,citi_bike_lat,citi_bike_lon,capacity,region_id,nearest_mta_station,meters_to_nearest_mta_station,current_dock_pct,future_num_docks_available
2094,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 03:53:40,2026-03-22 03:56:42,3,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0
4217,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 04:05:48,2026-03-22 04:07:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,6.0
6589,00284700-9d22-42ce-8485-113fed9879c1,12,6,2026-03-22 04:34:07,2026-03-22 04:37:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.315789,8.0
8777,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:04:29,2026-03-22 05:07:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0
11360,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:36:51,2026-03-22 05:37:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0


In [47]:
# Create future snapshot time
station_data_merged["future_snapshot_time"] = (
    station_data_merged.groupby("station_id")["snapshot_time"].shift(-1)
)

station_data_merged.head(5)

,station_id,num_bikes_available,num_docks_available,timestamp,snapshot_time,snapshot_hr,snapshot_weekday,is_weekend,citi_bike_station_name,citi_bike_lat,citi_bike_lon,capacity,region_id,nearest_mta_station,meters_to_nearest_mta_station,current_dock_pct,future_num_docks_available,future_snapshot_time
2094,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 03:53:40,2026-03-22 03:56:42,3,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0,2026-03-22 04:07:35
4217,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 04:05:48,2026-03-22 04:07:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,6.0,2026-03-22 04:37:35
6589,00284700-9d22-42ce-8485-113fed9879c1,12,6,2026-03-22 04:34:07,2026-03-22 04:37:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.315789,8.0,2026-03-22 05:07:35
8777,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:04:29,2026-03-22 05:07:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0,2026-03-22 05:37:35
11360,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:36:51,2026-03-22 05:37:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0,2026-03-22 06:07:35


In [48]:
# Future dock availability percentage
station_data_merged["future_dock_pct"] = (
    station_data_merged["future_num_docks_available"] / station_data_merged["capacity"]
)

In [49]:
# Check impossible future dock percentages
(station_data_merged["future_dock_pct"] > 1).sum()

0

In [50]:
# Drop rows where future dock count exceeds station capacity
station_data_merged = station_data_merged[
    station_data_merged["future_num_docks_available"] <= station_data_merged["capacity"]
].copy()

In [51]:
# Preview first five rows with new transformation
station_data_merged.head()

,station_id,num_bikes_available,num_docks_available,timestamp,snapshot_time,snapshot_hr,snapshot_weekday,is_weekend,citi_bike_station_name,citi_bike_lat,citi_bike_lon,capacity,region_id,nearest_mta_station,meters_to_nearest_mta_station,current_dock_pct,future_num_docks_available,future_snapshot_time,future_dock_pct
2094,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 03:53:40,2026-03-22 03:56:42,3,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0,2026-03-22 04:07:35,0.421053
4217,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 04:05:48,2026-03-22 04:07:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,6.0,2026-03-22 04:37:35,0.315789
6589,00284700-9d22-42ce-8485-113fed9879c1,12,6,2026-03-22 04:34:07,2026-03-22 04:37:35,4,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.315789,8.0,2026-03-22 05:07:35,0.421053
8777,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:04:29,2026-03-22 05:07:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0,2026-03-22 05:37:35,0.421053
11360,00284700-9d22-42ce-8485-113fed9879c1,10,8,2026-03-22 05:36:51,2026-03-22 05:37:35,5,6,1,28 Ave & 44 St,40.764089,-73.910651,19.0,71,46 St,893.80959,0.421053,8.0,2026-03-22 06:07:35,0.421053


In [52]:
# Check whether shifted row is actually about 30 minutes later
station_data_merged["minutes_to_future"] = (
    station_data_merged["future_snapshot_time"] - station_data_merged["snapshot_time"]
).dt.total_seconds() / 60

station_data_merged["minutes_to_future"].value_counts().sort_index()

minutes_to_future
10.883333        2193
29.983333      109703
30.000000      498132
30.016667      109701
60.000000           7
60.016667           2
90.000000           2
119.983333          1
120.000000          1
240.000000          1
300.000000          1
360.000000          1
360.016667          1
419.983333          1
480.000000          2
510.000000          1
630.000000          1
690.000000          1
899.983333          1
930.000000          1
960.000000          1
989.983333          1
1170.000000         1
3000.000000         1
Name: count, dtype: int64

In [53]:
# Drop rows where minutes to future is less than 25 or greater than 35
station_data_merged = station_data_merged[
    (station_data_merged["minutes_to_future"] >= 25) &
    (station_data_merged["minutes_to_future"] <= 35)
].copy()

station_data_merged.shape

(717536, 20)

In [54]:
# Drop rows with no future
station_data_merged = station_data_merged.dropna(
    subset=["future_num_docks_available", "future_snapshot_time", "future_dock_pct"]
).copy()

station_data_merged.shape

(717536, 20)

#### Create target column

This section defines the binary target for the modeling project.

The target, `lda_30min`, indicates whether a station is expected to have low dock availability approximately 30 minutes in the future. A positive case is defined as future dock availability at or below 10% of total station capacity.

This target creates a clear operational framing for the project:
predicting stations that may become constrained in the near future.

In [55]:
# Create target
station_data_merged["lda_30min"] = (
    station_data_merged["future_dock_pct"] <= 0.10
).astype(int)

station_data_merged["lda_30min"].value_counts()

lda_30min
0    607779
1    109757
Name: count, dtype: int64

#### Check data types before export

This section performs a final validation of data types and missing values before the cleaned dataset is passed to the next notebook.

Doing this check at the end of the acquisition/preprocessing stage helps confirm that the dataset is ready for exploratory analysis and reduces the risk of carrying forward schema issues into later project stages.

In [56]:
station_data_merged_types = pd.DataFrame()
station_data_merged_types["Data Type"] = station_data_merged.dtypes
station_data_merged_types["Count of Nulls"] = station_data_merged.isna().sum()
station_data_merged_types

,Data Type,Count of Nulls
station_id,object,0
num_bikes_available,int64,0
num_docks_available,int64,0
timestamp,datetime64[ns],0
snapshot_time,datetime64[ns],0
snapshot_hr,int32,0
snapshot_weekday,int32,0
is_weekend,int64,0
citi_bike_station_name,object,0
citi_bike_lat,float64,0


#### Drop unneeded columns

This section removes columns that are no longer needed for downstream analysis or modeling.

The goal is to keep the exported dataset focused on the variables that are most relevant for exploratory analysis, feature engineering, and prediction, while excluding intermediate fields used only during preprocessing and target construction.

In [57]:
station_data_merged = station_data_merged.drop(columns=[
    "timestamp",
    "nearest_mta_station",
    "region_id",
    "is_weekend"
]).copy()

station_data_merged.columns

Index(['station_id', 'num_bikes_available', 'num_docks_available',
       'snapshot_time', 'snapshot_hr', 'snapshot_weekday',
       'citi_bike_station_name', 'citi_bike_lat', 'citi_bike_lon', 'capacity',
       'meters_to_nearest_mta_station', 'current_dock_pct',
       'future_num_docks_available', 'future_snapshot_time', 'future_dock_pct',
       'minutes_to_future', 'lda_30min'],
      dtype='object')

#### Export for exploratory analysis

This section saves the cleaned and merged station-level dataset for use in the next phase of the project.

At this point, the data has been:
- assembled from multiple sources
- cleaned and validated
- enriched with transit-access information
- structured for short-horizon prediction

The exported file will serve as the input to the exploratory analysis notebook.

In [58]:
station_data_merged.to_parquet(
    CLEAN_DATA_DIR / "01_citi_bike_prediction_preprocessing.parquet",
    index=False
)